# MST-GNN Experiment A — CSI300 + VN30**Run All** to execute everything automatically. Results are:1. Printed inline in cell output2. Saved as JSON + pushed to GitHub3. Plotted (learning curves + backtest)| Dataset | Period | Est. Runtime ||---------|--------|--------------|| CSI300 | 2018→2026 | ~1h || VN30 | 2020→2026 | ~30min |

---## Cell 1 — Setup (GPU + Clone + Install + Git)

In [ ]:
import torch, subprocess, os, sys, time, glob, json

# --- GPU Check ---
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — will be slow!")

assert torch.cuda.is_available(), "GPU required!"

# --- Clone/Pull ---
REPO_URL = "https://github.com/quocnguyen5/mst-gnn-impl.git"
WORK_DIR = "/kaggle/working/mst-gnn-impl"

if os.path.exists(WORK_DIR):
    print("\nRepo exists — pulling latest...")
    !cd {WORK_DIR} && git pull --rebase
else:
    print("\nCloning repo...")
    !git clone {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)
!git log --oneline -3

# --- Install deps ---
!pip install -q torch_geometric akshare vnstock jieba pyarrow fastparquet tqdm 2>&1 | tail -3

# --- Git token ---
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.environ["GITHUB_TOKEN"] = token
    !git remote set-url origin https://{token}@github.com/quocnguyen5/mst-gnn-impl.git
    !git config user.email "nguyennq.app@gmail.com"
    !git config user.name "Kaggle AutoPush"
    print("✅ GitHub auto-push enabled.")
except Exception as e:
    os.environ["GITHUB_TOKEN"] = ""
    print(f"⚠️  No GitHub token ({e})")

print("\n✅ Setup complete!")


---## Cell 2 — Helper Functions

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

def run_experiment(cmd_args, label):
    """Run experiment subprocess and stream output."""
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable] + cmd_args,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=WORK_DIR, env={**os.environ},
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    ok = proc.returncode == 0
    status = "✅ SUCCESS" if ok else f"❌ FAILED (code {proc.returncode})"
    print(f"\n{label} {status} — {elapsed:.1f} min")
    return ok

def show_results(json_path, label):
    """Display results from JSON file."""
    print(f"\n{'═'*50}")
    print(f"  {label} TEST RESULTS")
    print(f"{'═'*50}")
    if not os.path.exists(json_path):
        print(f"  ❌ Results file not found: {json_path}")
        return None
    with open(json_path) as f:
        data = json.load(f)
    m = data["test_metrics"]
    print(f"  Best Epoch:  {data['best_epoch']}")
    print(f"  Epochs Run:  {data['total_epochs']}")
    for k, v in m.items():
        print(f"  {k:12s}: {v:.4f}")
    return data

def plot_curves(data, title, save_path):
    """Plot learning curves from results JSON data."""
    if not data or not data.get("train_history"):
        print("  No training history to plot.")
        return
    train_h = data["train_history"]
    val_h = data["val_history"]
    epochs = list(range(1, len(train_h) + 1))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for ax, key, ylabel in zip(axes, ['loss', 'accuracy', 'damrr'], ['Loss', 'Accuracy', 'DAMRR']):
        ax.plot(epochs, [m[key] for m in train_h], 'b-', label='Train', alpha=0.8)
        ax.plot(epochs, [m[key] for m in val_h], 'r-', label='Val', alpha=0.8)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")

def show_backtest(chart_path):
    """Display backtest cumulative returns chart."""
    if os.path.exists(chart_path):
        display(IPImage(filename=chart_path))
    else:
        print(f"  Chart not found: {chart_path}")


---## Cell 3 — Run CSI300 + Show Results

In [ ]:
# === RUN CSI300 ===
ok = run_experiment(["-m", "experiments.run_main", "--dataset", "csi300", "--aggregator", "mean"], "CSI300")

# === SHOW RESULTS ===
if ok:
    data = show_results("checkpoints/results_csi300_mean.json", "CSI300")
    plot_curves(data, "CSI300 Learning Curves", "checkpoints/csi300_curves.png")
    show_backtest("checkpoints/cumulative_returns_csi300.png")
else:
    print("\n❌ Experiment failed! Check output above.")


---## Cell 4 — Run VN30 + Show Results

In [ ]:
# === RUN VN30 ===
ok = run_experiment(["-m", "experiments.run_vn", "--universe", "vn30", "--aggregator", "mean", "--start", "2020-01-02", "--end", "2026-06-26"], "VN30")

# === SHOW RESULTS ===
if ok:
    data = show_results("checkpoints/results_vn30_mean.json", "VN30")
    plot_curves(data, "VN30 Learning Curves", "checkpoints/vn30_curves.png")
    show_backtest("checkpoints/cumulative_returns_vn30.png")
else:
    print("\n❌ Experiment failed! Check output above.")


---## Cell 5 — Final Comparison: CSI300 vs VN30

In [ ]:
import pandas as pd

results = {}
for ds_id, label in [("csi300", "CSI300"), ("vn30", "VN30")]:
    path = f"checkpoints/results_{ds_id}_mean.json"
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        results[label] = data["test_metrics"]

if results:
    df = pd.DataFrame(results).T
    df.index.name = 'Dataset'
    print("\n" + "═" * 60)
    print("  FINAL COMPARISON — MST-GNN (Mean Aggregator)")
    print("═" * 60)
    print(df.to_string())
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    cols = [c for c in ['accuracy', 'precision', 'damrr'] if c in df.columns]
    df[cols].plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white', width=0.7)
    ax.set_title('MST-GNN: CSI300 vs VN30', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score'); ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("checkpoints/comparison_A.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No results found — run Cells 3 and 4 first.")

# === File listing ===
print("\n=== Checkpoints ===")
for f in sorted(glob.glob("checkpoints/*")):
    sz = os.path.getsize(f) / 1e6
    print(f"  {f} ({sz:.1f} MB)")


---## Cell 6 — Force Reset (Optional)Uncomment to delete all checkpoints and re-train from scratch.

In [ ]:
# ⚠️ UNCOMMENT TO RESET
# import glob, os
# for f in glob.glob("checkpoints/*.pt") + glob.glob("checkpoints/*.json"):
#     os.remove(f); print(f"Removed: {f}")
# print("✅ Reset complete. Re-run from Cell 3.")
